# Front-door, instruments, and the linear estimators

The graph names the route; these are the estimators for each. All are pure numpy with
classical standard errors, return a `LinearEstimate` spec, and their intervals are labelled
`wald` — a frequentist CI is a different object from a credible interval and the type says so.

In [ ]:
from axiom.identify import (
    CausalGraph, EndogeneityTest, FrontDoorRoute, InstrumentRoute, LinearEstimate,
    conditional_instruments, durbin_wu_hausman, frontdoor_admissible, frontdoor_linear,
    frontdoor_sets, hausman_iv_vs_ols, identify, instrument_admissible, instruments, ols,
    two_stage_least_squares, weak_instrument_check,
)
from axiom.sim import confounded_world, frontdoor_world, iv_world

## Front-door criterion

In [ ]:
fd_graph = CausalGraph.from_edges("X -> M, M -> Y, X <-> Y")
print(frontdoor_admissible(fd_graph, "X", "Y", ["M"]), frontdoor_sets(fd_graph, "X", "Y"))
route = FrontDoorRoute(mediators=("M",), treatment="X", outcome="Y")
print(route)

# a latent between X and M breaks condition (ii)
print(frontdoor_admissible(CausalGraph.from_edges("X -> M, M -> Y, X <-> Y, X <-> M"), "X", "Y", ["M"]))

## Instruments

`instruments` lists unconditional instruments (relevance in $G$, exclusion in $G_{\underline{X}}$);
`conditional_instruments` lists $(Z, W)$ pairs where the exclusion holds only given $W$.

In [ ]:
iv_graph = CausalGraph.from_edges("Z -> X, X -> Y, X <-> Y")
print(instruments(iv_graph, "X", "Y"), instrument_admissible(iv_graph, "X", "Y", "Z"))
print(instruments(CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y, X <-> Y"), "X", "Y"), "<- exclusion violated")

civ = CausalGraph.from_edges("W -> Z, W -> Y, Z -> X, X -> Y, X <-> Y")
print(instruments(civ, "X", "Y"), conditional_instruments(civ, "X", "Y"))
print(InstrumentRoute(instrument="Z", conditioning=("W",), treatment="X", outcome="Y"))

## Estimating along the licensed route

Each `sim` world carries its truth. The naive estimate is biased by construction; the
estimator matching the verdict's route recovers the truth within its standard error.

In [ ]:
world = confounded_world()
frame = world.observed(world.simulate(20_000, seed=0))
truth = world.total_effect("X", "Y")
v = identify(world.graph, "X", "Y")

naive: LinearEstimate = ols(frame, "Y", "X")
adjusted = ols(frame, "Y", "X", covariates=v.adjustment_set)
print(f"truth {truth} | naive {naive.estimate:.3f} ± {naive.se:.3f} | adjusted {adjusted.estimate:.3f} ± {adjusted.se:.3f}")
print(adjusted.ci(0.95), adjusted.method, adjusted.covariates)

In [ ]:
world = iv_world()
frame = world.observed(world.simulate(20_000, seed=1))
v = identify(world.graph, "X", "Y")
iv = two_stage_least_squares(frame, "Y", "X", instruments=[v.instrument])
print(f"truth {world.total_effect('X', 'Y')} | ols {ols(frame, 'Y', 'X').estimate:.3f} | 2sls {iv.estimate:.3f} ± {iv.se:.3f}")
print(iv.detail)
strength = weak_instrument_check(iv)
print(strength.name, strength.state, strength.statement)

In [ ]:
world = frontdoor_world()
frame = world.observed(world.simulate(20_000, seed=2))
v = identify(world.graph, "X", "Y")
fd = frontdoor_linear(frame, "Y", "X", mediators=v.mediators)
print(f"truth {world.total_effect('X', 'Y')} | ols {ols(frame, 'Y', 'X').estimate:.3f} | front-door {fd.estimate:.3f} ± {fd.se:.3f}")
print(fd.ci(0.9))

## Is the IV route needed? Endogeneity tests

`durbin_wu_hausman` is the control-function form; `hausman_iv_vs_ols` contrasts the two
estimates directly. A degenerate contrast (negative variance difference) returns a typed
`Unverified`, not a fabricated p-value.

In [ ]:
world = iv_world()
frame = world.observed(world.simulate(20_000, seed=1))
dwh = durbin_wu_hausman(frame, "Y", "X", instruments=["Z"])
print(dwh if not isinstance(dwh, EndogeneityTest) else (dwh.conclusion, round(dwh.statistic, 2), dwh.p_value))

h = hausman_iv_vs_ols(ols(frame, "Y", "X"), two_stage_least_squares(frame, "Y", "X", instruments=["Z"]))
print(h if not isinstance(h, EndogeneityTest) else (h.conclusion, round(h.statistic, 2)))

## Guard rails

The estimators refuse the outcome on the right-hand side, missing columns, and too few rows —
loudly, never with a confidently wrong number.

In [ ]:
for bad in (lambda: ols(frame, "Y", "X", covariates=["Y"]), lambda: ols(frame, "Y", "nope"), lambda: ols(frame.head(2), "Y", "X")):
    try:
        bad()
    except (ValueError, KeyError) as e:
        print(type(e).__name__, "->", str(e)[:90])